In [1]:
# =========================================================
# 1) IMPORTS
# =========================================================

import pyodbc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# =========================================================
# 2) LEITURA DA VIEW DO SQL SERVER
# =========================================================

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=.\SQLEXPRESS;"
    "DATABASE=Walmart_Anomaly_Detection;"
    "Trusted_Connection=yes;"
)

query = """
SELECT *
FROM dbo.vw_orders_features_for_ml
"""

df = pd.read_sql(query, conn)
conn.close()

print("Shape inicial:", df.shape)
display(df.head())

# =========================================================
# 3) PREPARAÇÃO MÍNIMA
# =========================================================

df = df.copy()

if "order_id" in df.columns:
    df = df.drop_duplicates(subset=["order_id"])

df = df[df["has_missing"].notna()].copy()
df["has_missing"] = df["has_missing"].astype(int)

if "order_amount_num" in df.columns and "order_amount" not in df.columns:
    df["order_amount"] = df["order_amount_num"]

if "items_total" not in df.columns and {"items_delivered_num", "items_missing_num"}.issubset(df.columns):
    df["items_total"] = df["items_delivered_num"] + df["items_missing_num"]

if "delivery_hour_only" not in df.columns and "delivery_hour" in df.columns:
    df["delivery_hour_only"] = pd.to_datetime(df["delivery_hour"], errors="coerce").dt.hour

# =========================================================
# 4) FEATURES DERIVADAS
# =========================================================

df["high_value_order"] = (
    df["order_amount"] >= df["order_amount"].quantile(0.75)
).astype(int)

risk_hours = [6, 7, 10, 13, 22]
df["night_delivery"] = df["delivery_hour_only"].isin(risk_hours).astype(int)

# =========================================================
# 5) REMOVER COLUNAS AGREGADAS DA VIEW QUE PODEM GERAR LEAKAGE
# E RECALCULAR OOF NO NOTEBOOK
# =========================================================

cols_to_drop_if_exist = [
    "driver_problem_rate",
    "region_problem_rate",
    "customer_problem_rate",
    "driver_orders_count",
    "customer_orders_count",
    "driver_avg_order_amount",
    "driver_total_orders",
    "driver_problem_rate_oof",
    "driver_total_orders_oof"
]

for col in cols_to_drop_if_exist:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

# =========================================================
# 6) SPLIT PRINCIPAL
# IMPORTANTE: OOF SERÁ CALCULADO APENAS NO TREINO
# =========================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    stratify=df["has_missing"],
    random_state=42
)

train_df = train_df.copy()
test_df = test_df.copy()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

# =========================================================
# 7) DRIVER HISTORY OOF APENAS NO TREINO
# =========================================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_df["driver_problem_rate_oof"] = np.nan
train_df["driver_total_orders_oof"] = np.nan

for tr_idx, val_idx in skf.split(train_df, train_df["has_missing"]):

    tr_fold = train_df.iloc[tr_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()

    driver_stats = (
        tr_fold
        .groupby("driver_id")
        .agg(
            driver_total_orders=("order_id", "count"),
            driver_problem_rate=("has_missing", "mean")
        )
        .reset_index()
    )

    val_enriched = val_fold.merge(driver_stats, on="driver_id", how="left")

    train_df.loc[train_df.index[val_idx], "driver_problem_rate_oof"] = val_enriched["driver_problem_rate"].values
    train_df.loc[train_df.index[val_idx], "driver_total_orders_oof"] = val_enriched["driver_total_orders"].values

train_df["driver_problem_rate_oof"] = train_df["driver_problem_rate_oof"].fillna(train_df["has_missing"].mean())
train_df["driver_total_orders_oof"] = train_df["driver_total_orders_oof"].fillna(1)

# =========================================================
# 8) DRIVER HISTORY PARA O TESTE
# CALCULADO SOMENTE A PARTIR DO TREINO
# =========================================================

driver_stats_full_train = (
    train_df
    .groupby("driver_id")
    .agg(
        driver_total_orders=("order_id", "count"),
        driver_problem_rate=("has_missing", "mean")
    )
    .reset_index()
)

test_df = test_df.merge(driver_stats_full_train, on="driver_id", how="left")

test_df.rename(columns={
    "driver_problem_rate": "driver_problem_rate_oof",
    "driver_total_orders": "driver_total_orders_oof"
}, inplace=True)

test_df["driver_problem_rate_oof"] = test_df["driver_problem_rate_oof"].fillna(train_df["has_missing"].mean())
test_df["driver_total_orders_oof"] = test_df["driver_total_orders_oof"].fillna(1)

# =========================================================
# 9) FEATURES
# IGUAIS À VERSÃO ANTERIOR
# =========================================================

feature_cols = [
    "order_amount",
    "items_total",
    "delivery_hour_only",
    "night_delivery",
    "high_value_order",
    "driver_problem_rate_oof",
    "driver_total_orders_oof",
    "region"
]

X_train = train_df[feature_cols].copy()
y_train = train_df["has_missing"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["has_missing"].copy()

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# =========================================================
# 10) COMPARAÇÃO DE MODELOS
# =========================================================

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced_subsample",
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(),
    "SVM": SVC(
        probability=True,
        class_weight="balanced"
    )
}

results = []

for name, model in models.items():

    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    roc = roc_auc_score(y_test, proba)
    prec, rec, _ = precision_recall_curve(y_test, proba)
    pr_auc = auc(rec, prec)

    print("\n===============================")
    print(name)
    print("===============================")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))
    print("ROC-AUC:", round(roc, 4))
    print("PR-AUC:", round(pr_auc, 4))

    results.append({
        "Model": name,
        "ROC_AUC": roc,
        "PR_AUC": pr_auc
    })

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
display(results_df)

# =========================================================
# 11) MODELO FINAL
# =========================================================

best_model = Pipeline([
    ("prep", preprocess),
    ("model", GradientBoostingClassifier())
])

best_model.fit(X_train, y_train)

# =========================================================
# 12) SCORE SOMENTE NO TESTE
# =========================================================

test_df["risk_probability"] = best_model.predict_proba(X_test)[:, 1]

def risk_level(p):
    if p > 0.6:
        return "High Risk"
    elif p > 0.3:
        return "Medium Risk"
    else:
        return "Low Risk"

test_df["risk_level"] = test_df["risk_probability"].apply(risk_level)

# =========================================================
# 13) AVALIAÇÃO FINAL NO TESTE
# =========================================================

print("\nDistribuição de risk_level no teste:")
print(test_df["risk_level"].value_counts(dropna=False))

print("\nResumo de risk_probability no teste:")
print(test_df["risk_probability"].describe())

print("\nTaxa de has_missing por nível de risco (teste):")
print(test_df.groupby("risk_level")["has_missing"].mean())

# =========================================================
# 14) BASE FINAL PARA DASHBOARD
# APENAS TESTE
# =========================================================

dashboard_df = test_df[[
    "order_id",
    "risk_probability",
    "risk_level",
    "order_amount",
    "region",
    "driver_id",
    "delivery_hour_only",
    "has_missing"
]].copy()

dashboard_df["risk_probability"] = dashboard_df["risk_probability"].round(4)

display(dashboard_df.head())

# =========================================================
# 15) EXPORTAÇÃO
# =========================================================

dashboard_df.to_csv("orders_scored_test_only.csv", index=False)
dashboard_df.to_excel("orders_scored_test_only.xlsx", index=False)

print("\nArquivos salvos com sucesso:")
print("- orders_scored_test_only.csv")
print("- orders_scored_test_only.xlsx")

C:\Users\erika\AppData\Local\Temp\ipykernel_39136\347601841.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Shape inicial: (10000, 23)


,order_id,order_date,order_month,order_amount_num,region,items_delivered_num,items_missing_num,items_total,missing_rate,has_missing,...,driver_age,driver_total_trips_2023,customer_id,customer_age,driver_problem_rate,region_problem_rate,customer_problem_rate,driver_orders_count,customer_orders_count,driver_avg_order_amount
0,7e5317f0-3eaa-48d6-be75-4f6091931043,2023-01-20,1,3735.0,Altamonte Springs,16,0,16,0.0,0,...,24,60,WCID5000,38,0.000000,0.161992,0.0,7,7,32233.142857
1,052ef301-b6a1-4e08-a6eb-8c92f64c11af,2023-02-12,2,11350.0,Clermont,8,0,8,0.0,0,...,60,25,WCID5000,38,0.000000,0.158237,0.0,6,7,26371.333333
2,ddc396c6-db33-4caf-8f2a-4bf1e7b486df,2023-03-06,3,25636.0,Clermont,16,0,16,0.0,0,...,24,50,WCID5000,38,0.000000,0.158237,0.0,7,7,27898.142857
3,ab3306b1-e6e9-4cc2-ba97-741406bdd448,2023-04-18,4,47256.0,Winter Park,7,0,7,0.0,0,...,62,19,WCID5000,38,0.272727,0.144781,0.0,11,7,24805.909091
4,578f5e59-cb24-4a28-b42a-8cf4a189ca83,2023-06-08,6,35614.0,Orlando,14,0,14,0.0,0,...,51,31,WCID5000,38,0.272727,0.151320,0.0,11,7,35742.727273


Train: (7500, 20)
Test : (2500, 20)

Logistic Regression
[[1334  791]
 [  22  353]]
              precision    recall  f1-score   support

           0       0.98      0.63      0.77      2125
           1       0.31      0.94      0.46       375

    accuracy                           0.67      2500
   macro avg       0.65      0.78      0.62      2500
weighted avg       0.88      0.67      0.72      2500

ROC-AUC: 0.7988
PR-AUC: 0.3094

Decision Tree
[[1550  575]
 [  25  350]]
              precision    recall  f1-score   support

           0       0.98      0.73      0.84      2125
           1       0.38      0.93      0.54       375

    accuracy                           0.76      2500
   macro avg       0.68      0.83      0.69      2500
weighted avg       0.89      0.76      0.79      2500

ROC-AUC: 0.8893
PR-AUC: 0.5916

Random Forest
[[2060   65]
 [ 262  113]]
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      2125
       

,Model,ROC_AUC,PR_AUC
3,Gradient Boosting,0.917994,0.654843
1,Decision Tree,0.889298,0.591589
4,SVM,0.888797,0.481519
2,Random Forest,0.884612,0.565945
0,Logistic Regression,0.798832,0.309368



Distribuição de risk_level no teste:
risk_level
Low Risk       1944
Medium Risk     466
High Risk        90
Name: count, dtype: int64

Resumo de risk_probability no teste:
count    2500.000000
mean        0.161715
std         0.191689
min         0.002097
25%         0.020481
50%         0.063214
75%         0.273996
max         0.987889
Name: risk_probability, dtype: float64

Taxa de has_missing por nível de risco (teste):
risk_level
High Risk      0.866667
Low Risk       0.049383
Medium Risk    0.431330
Name: has_missing, dtype: float64


,order_id,risk_probability,risk_level,order_amount,region,driver_id,delivery_hour_only,has_missing
0,bbe76204-9d86-4981-a906-76adff39df49,0.1306,Low Risk,33295.0,Altamonte Springs,WDID10197,5,0
1,81960888-bd0f-4fd1-b4f7-5b76f664dcae,0.0165,Low Risk,7351.0,Kissimmee,WDID10126,17,0
2,419cd643-fd20-4b18-9332-18c491a9c345,0.0276,Low Risk,31164.0,Altamonte Springs,WDID10770,17,0
3,69371e88-e68c-4522-8f0f-270ccbddf136,0.0167,Low Risk,14217.0,Winter Park,WDID10775,0,0
4,036b5af2-8fd6-4e61-b11a-d7ba7cd453b4,0.0119,Low Risk,40002.0,Orlando,WDID10110,6,0



Arquivos salvos com sucesso:
- orders_scored_test_only.csv
- orders_scored_test_only.xlsx
